# CPM-Ridge cross-group validation

本 notebook 做 highIAT 与 lowIAT 的双向跨组验证。核心原则是：

1. 在 source 组内部用 **LOOCV** 选择最终 `ridge_alpha`；
2. 用整个 source 组重新估计协变量残差化、CPM 选边、训练集标准化和 Ridge 模型；
3. target 组只接受 source 已锁定的变换和模型，不参与选边、调参或模型拟合。

代码尽量复用 `biox` 中 CPM 模块的函数。这里使用了 CPM 模块内部函数，是为了让最终模型的残差化、选边、标准化和预测步骤与 `cpm_nested_cv()` 完全一致。


In [1]:
# ============================================================
# 0. 线程控制：必须在 numpy/scipy/sklearn 大量调用前设置
# ============================================================
import os

MAX_BLAS_THREADS = 1
os.environ["OMP_NUM_THREADS"] = str(MAX_BLAS_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(MAX_BLAS_THREADS)
os.environ["MKL_NUM_THREADS"] = str(MAX_BLAS_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(MAX_BLAS_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(MAX_BLAS_THREADS)

print("MAX_BLAS_THREADS=", MAX_BLAS_THREADS)


MAX_BLAS_THREADS= 1


In [2]:
# ============================================================
# 1. Imports：尽量复用 DATA_ROOT/biox 中的 CPM、FC 工具
# ============================================================
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# 数据、时间序列和 biox package 都位于 DATA_ROOT 下。
# Path.home() 方便在 /Users/yiling、/Users/fmri24 等不同账号间切换。
DATA_ROOT = Path.home() / "Desktop" / "YiZhang"
if str(DATA_ROOT) not in sys.path:
    sys.path.insert(0, str(DATA_ROOT))

from biox.CPM import (
    _expand_param_grid,
    _extract_fold_edge_record,
    _fit_and_predict_prepared_fold,
    _score_predictions,
    _select_and_prepare_fold_features,
    _select_best_hyperparams_inner_cv,
    _validate_model_params,
    calculate_permutation_p_value,
    calculate_prediction_performance,
    fast_partCorr_train,
)
from biox.fcmat import fc_matrix_to_edges
from biox.prep import fcMatrixGp

print("DATA_ROOT:", DATA_ROOT)
print("Imported biox.CPM helpers, biox.fcmat.fc_matrix_to_edges, and biox.prep.fcMatrixGp")


DATA_ROOT: /Users/fmri24/Desktop/YiZhang
Imported biox.CPM helpers, biox.fcmat.fc_matrix_to_edges, and biox.prep.fcMatrixGp


## 2. 路径与参数

参数尽量与 `CPM_nestedCV-LOOCV.ipynb` 保持一致。区别是：本 notebook 不再做 nested outer CV，而是在 source 组内部用 LOOCV 锁定最终 `ridge_alpha`，再把最终模型外推到 target 组。


In [3]:
# ============================================================
# 2. 路径与全局参数
# ============================================================
SCALE_CSV = DATA_ROOT / "量表数据汇总_with_MeanEnorm.csv"
SCHAEFER_DIR = DATA_ROOT / "HBN_func_Schaefer200"
HO_DIR = DATA_ROOT / "HBN_func_HarvardOxfordSubcortical"

# Harvard-Oxford 原始 ROI 下标；原编号为 1-based，因此这里减 1。
HO_RAW_INDEX = np.array([4, 5, 6, 7, 9, 10, 11, 15, 16, 17, 18, 19, 20, 21]) - 1
EXPECTED_N_TIMEPOINTS = 740

IAT_LABEL = "IAT_PCIAT_Total"
SITE_COL = "ScannerSite"
CBCL_TOTAL_COL = "CBCL,CBCL_Total_T"
MOTION_COL = "mean_enorm"
COVARIATE_COLS = [
    "Basic_Demos,Age",
    "Basic_Demos,Sex",
    "Barratt,Barratt_Total_Edu",
    "Barratt,Barratt_Total_Occ",
    "CBCL,CBCL_Int_T",
    "CBCL,CBCL_Ext_T",
    MOTION_COL,
]

IAT_CUTOFF = 30
CBCL_CUTOFF = 70
ENORM_CUTOFF = 0.3

# CPM-Ridge 参数。
P_THRESH = 0.01
RANDOM_STATE = 2026
N_JOBS = 20
STANDARDIZE_FEATURES = True

# 用 source 组内部 LOOCV 锁定最终 alpha。
LOCKING_INNER_CV_TYPE = "loocv"
LOCKING_INNER_N_SPLITS = 5  # LOOCV 时该参数会被 biox 忽略；保留只是为了函数签名完整。
SCORING = "pearson_r"

# Ridge alpha 网格与主 LOOCV 脚本保持一致。
ALPHA_GRID = np.logspace(-2, 2, 100)
ALPHA_GRID = np.sort(ALPHA_GRID.astype(float))

# target 组 inference：固定 source 模型，只重排 target 的真实残差。
N_PERMUTATIONS = 5000
N_BOOTSTRAP = 5000

RESULT_DIR = DATA_ROOT / "CPM_nestedCV_crossGroup"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("SCALE_CSV:", SCALE_CSV)
print("RESULT_DIR:", RESULT_DIR)
print("n_alpha:", len(ALPHA_GRID), "alpha range:", (ALPHA_GRID.min(), ALPHA_GRID.max()))


SCALE_CSV: /Users/fmri24/Desktop/YiZhang/量表数据汇总_with_MeanEnorm.csv
RESULT_DIR: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_crossGroup
n_alpha: 100 alpha range: (np.float64(0.01), np.float64(100.0))


## 3. FC 计算

这一节模仿 `CPM_nestedCV-LOOCV.ipynb`：每名被试读取 `func1` 和 `func2`，拼接 Schaefer200 与 HO14 时间序列；`biox.prep.fcMatrixGp()` 计算 Fisher-z FC 矩阵，`biox.fcmat.fc_matrix_to_edges()` 提取上三角 edge。


In [4]:
# ============================================================
# 3. FC 计算 / 数据读取
# ============================================================
# 不在这里依赖额外 cache；按主 LOOCV 脚本的过程直接从原始时间序列计算 FC。
df_atlas = pd.read_csv(SCALE_CSV)
df_atlas["ID"] = df_atlas["ID"].astype(str)
df_atlas["func1"] = "sub-" + df_atlas["ID"] + "_func1.npy"
df_atlas["func2"] = "sub-" + df_atlas["ID"] + "_func2.npy"

# 只有两个 atlas 目录中都存在 func1/func2 的被试，才能拼接 cortical + subcortical ROI。
files_present_in_both_atlases = set(os.listdir(SCHAEFER_DIR)) & set(os.listdir(HO_DIR))
df_atlas = df_atlas[
    df_atlas["func1"].isin(files_present_in_both_atlases)
    & df_atlas["func2"].isin(files_present_in_both_atlases)
].reset_index(drop=True)

fc_list = []
timepoint_ok_row_idx = []
for i in range(df_atlas.shape[0]):
    f1 = df_atlas.loc[i, "func1"]
    f2 = df_atlas.loc[i, "func2"]

    sch1 = np.load(SCHAEFER_DIR / f1)
    sch2 = np.load(SCHAEFER_DIR / f2)
    ho1 = np.load(HO_DIR / f1)
    ho2 = np.load(HO_DIR / f2)

    # 两个 run 拼接后的时间点数必须一致，否则该被试不能进入 FC 矩阵。
    if (sch1.shape[1] + sch2.shape[1] != EXPECTED_N_TIMEPOINTS) or (ho1.shape[1] + ho2.shape[1] != EXPECTED_N_TIMEPOINTS):
        continue

    ts_sch = np.concatenate([sch1, sch2], axis=1)
    ts_ho = np.concatenate([ho1, ho2], axis=1)[HO_RAW_INDEX, :]
    ts = np.concatenate([ts_sch, ts_ho], axis=0)

    fc_z = fcMatrixGp(ts[np.newaxis, :, :], fisherZ=True)[0]
    fc_list.append(fc_matrix_to_edges(fc_z, k=1))
    timepoint_ok_row_idx.append(i)

# df_atlas 第 i 行与 FC_all 第 i 行严格对应。
df_atlas = df_atlas.iloc[timepoint_ok_row_idx].reset_index(drop=True)
FC_all = np.asarray(fc_list, dtype=np.float32)
print("FC_all:", FC_all.shape)


/Users/fmri24/Desktop/YiZhang/biox/prep.py:623: RuntimeWarning: divide by zero encountered in arctanh
  fcMatrixGp = np.arctanh(fcMatrixGp)


FC_all: (1376, 22791)


## 4. 共同分析样本与协变量编码

跨组预测时，highIAT 与 lowIAT 必须使用完全相同的协变量列。这里先在完整分析样本上固定 site dummy 的类别和顺序，再切分 subgroup。


In [5]:
# ============================================================
# 4. 构建共同分析样本、共同 site dummy 和两个 subgroup
# ============================================================
df = df_atlas.copy()
for c in COVARIATE_COLS + [IAT_LABEL, CBCL_TOTAL_COL]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

analysis_ready_mask = (
    df[COVARIATE_COLS + [IAT_LABEL, CBCL_TOTAL_COL, SITE_COL]].notna().all(axis=1).to_numpy()
    & np.isfinite(df[IAT_LABEL].to_numpy(dtype=float))
    & np.isfinite(df[MOTION_COL].to_numpy(dtype=float))
    & (df[MOTION_COL].to_numpy(dtype=float) < ENORM_CUTOFF)
    & (df[CBCL_TOTAL_COL].to_numpy(dtype=float) < CBCL_CUTOFF)
    & np.isfinite(FC_all).all(axis=1)
)

df_model = df.loc[analysis_ready_mask].reset_index(drop=True)
X_model = FC_all[analysis_ready_mask].astype(np.float32, copy=False)
y_model = df_model[IAT_LABEL].to_numpy(dtype=float)

# 在完整分析样本上一次性建立 site dummy，确保两组的 dummy 列完全一致。
site_categories = sorted(df_model[SITE_COL].astype(str).unique().tolist())
site_cat = pd.Categorical(df_model[SITE_COL].astype(str), categories=site_categories)
site_dummies = pd.get_dummies(site_cat, prefix=SITE_COL, drop_first=True, dtype=float)

cov_basic_all = df_model[COVARIATE_COLS].to_numpy(dtype=float)
cov_all = np.column_stack([cov_basic_all, site_dummies.to_numpy(dtype=float)])
covariate_names = COVARIATE_COLS + list(site_dummies.columns)

low_mask = y_model <= IAT_CUTOFF
high_mask = y_model > IAT_CUTOFF

groups = {
    "lowIAT": {
        "X": X_model[low_mask],
        "y": y_model[low_mask],
        "C": cov_all[low_mask],
        "ID": df_model.loc[low_mask, "ID"].astype(str).to_numpy(),
    },
    "highIAT": {
        "X": X_model[high_mask],
        "y": y_model[high_mask],
        "C": cov_all[high_mask],
        "ID": df_model.loc[high_mask, "ID"].astype(str).to_numpy(),
    },
}

print("Site categories:", site_categories)
print("Covariate columns:", covariate_names)
for name, g in groups.items():
    print(
        f"{name}: N={g['y'].size}, edges={g['X'].shape[1]}, "
        f"covariates={g['C'].shape[1]}, IAT mean={g['y'].mean():.3f}, sd={g['y'].std():.3f}"
    )

assert groups["lowIAT"]["X"].shape[1] == groups["highIAT"]["X"].shape[1]
assert groups["lowIAT"]["C"].shape[1] == groups["highIAT"]["C"].shape[1]


Site categories: ['CBIC', 'CUNY', 'RU', 'RU_ex']
Covariate columns: ['Basic_Demos,Age', 'Basic_Demos,Sex', 'Barratt,Barratt_Total_Edu', 'Barratt,Barratt_Total_Occ', 'CBCL,CBCL_Int_T', 'CBCL,CBCL_Ext_T', 'mean_enorm', 'ScannerSite_CUNY', 'ScannerSite_RU', 'ScannerSite_RU_ex']
lowIAT: N=653, edges=22791, covariates=10, IAT mean=17.504, sd=8.921
highIAT: N=461, edges=22791, covariates=10, IAT mean=41.534, sd=9.244


## 5. Source 组 LOOCV 锁定参数并预测 target 组

`lock_alpha_with_source_loocv()` 只在 source 组内部运行 LOOCV，用来选择最终 `ridge_alpha`。随后 `fit_source_predict_target()` 用整个 source 组训练最终模型，并把 target 组作为完全外部样本预测。


In [6]:
# ============================================================
# 5. 工具函数：source 内 LOOCV 锁定 alpha，source 全样本拟合并预测 target
# ============================================================
SELECTION_PARAMS = {
    "selection_method": "pvalue",
    "p_thresh": P_THRESH,
    "top_k": None,
    "top_k_mode": "total",
    "max_edges_total": None,
    "min_edges_total": 0,
    "fallback_if_too_few": False,
    "max_edges_per_sign": None,
}


def lock_alpha_with_source_loocv(source):
    """仅使用 source 组，通过 biox 的 inner-LOOCV 逻辑选择最终 ridge_alpha。"""
    model_type, base_model_params = _validate_model_params(
        model_type="ridge",
        ridge_alpha=1.0,
        lasso_alpha=1.0,
        elasticnet_alpha=1.0,
        l1_ratio=0.5,
        pls_components=None,
        pca_components=None,
    )
    param_candidates, candidate_model_params = _expand_param_grid(
        {"ridge_alpha": ALPHA_GRID},
        model_type=model_type,
        base_model_params=base_model_params,
    )

    best_model_params, best_score, candidate_scores, inner_n_splits_actual = _select_best_hyperparams_inner_cv(
        X=source["X"],
        y=source["y"],
        Covariates=source["C"],
        param_candidates=param_candidates,
        candidate_model_params=candidate_model_params,
        model_type=model_type,
        inner_cv_type=LOCKING_INNER_CV_TYPE,
        inner_n_splits=LOCKING_INNER_N_SPLITS,
        random_state=RANDOM_STATE,
        selection_params=SELECTION_PARAMS,
        standardize_features=STANDARDIZE_FEATURES,
        scoring=SCORING,
    )
    return model_type, best_model_params, best_score, candidate_scores, inner_n_splits_actual


def fixed_prediction_permutation(y_true, y_pred, n_permutations, random_state):
    """固定 source 模型预测值，只重排 target 真实残差，得到 Pearson r 零分布。"""
    rng = np.random.default_rng(random_state)
    null_scores = np.empty(n_permutations, dtype=float)
    for perm_idx in range(n_permutations):
        null_scores[perm_idx] = _score_predictions(
            rng.permutation(y_true),
            y_pred,
            "pearson_r",
        )
    observed = _score_predictions(y_true, y_pred, "pearson_r")
    p_result = calculate_permutation_p_value(
        observed_score=observed,
        null_scores=null_scores,
        performance_metric="pearson_r",
        alternative="greater",
    )
    return observed, null_scores, p_result


def bootstrap_metric_ci(y_true, y_pred, n_bootstrap, random_state, confidence_level=0.95):
    """对 target 被试做 bootstrap，估计外部预测 Pearson r 的置信区间。"""
    rng = np.random.default_rng(random_state)
    n = len(y_true)
    values = np.empty(n_bootstrap, dtype=float)
    for b in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        values[b] = _score_predictions(y_true[idx], y_pred[idx], "pearson_r")

    alpha = 1.0 - confidence_level
    finite_values = values[np.isfinite(values)]
    return {
        "bootstrap_values": values,
        "ci_low": float(np.quantile(finite_values, alpha / 2)),
        "ci_high": float(np.quantile(finite_values, 1 - alpha / 2)),
        "n_valid_bootstrap": int(finite_values.size),
    }


def fit_source_predict_target(source_name, target_name):
    source = groups[source_name]
    target = groups[target_name]

    print("\n" + "=" * 80)
    print(f"SOURCE: {source_name}  ->  TARGET: {target_name}")
    print("=" * 80)

    # 1) 只在 source 组内部用 LOOCV 锁定 ridge_alpha。
    model_type, best_model_params, best_score, candidate_scores, inner_n_splits_actual = lock_alpha_with_source_loocv(source)
    print("Locked by source LOOCV:", best_model_params)
    print("Source LOOCV score:", best_score)

    # 2) 用整个 source 组锁定协变量残差化、选边、标准化，并准备 target 特征。
    prepared = _select_and_prepare_fold_features(
        X_train=source["X"],
        y_train=source["y"],
        Covariates_train=source["C"],
        X_test=target["X"],
        y_test=target["y"],
        Covariates_test=target["C"],
        model_type=model_type,
        standardize_features=STANDARDIZE_FEATURES,
        **SELECTION_PARAMS,
    )

    # 3) 在 source 组全部被试上拟合最终 Ridge，并预测 target 组残差 IAT。
    y_pred_target, fitted_model = _fit_and_predict_prepared_fold(
        prepared,
        model_type=model_type,
        model_params=best_model_params,
        random_state=RANDOM_STATE,
        return_model=True,
    )
    y_true_target = prepared["y_resid_true_test"]

    # 4) 额外保存 source 组 beta_y，便于追踪 target 残差的定义。
    _, _, _, _, _, beta_y_source = fast_partCorr_train(source["X"], source["y"], source["C"])

    edge_record = _extract_fold_edge_record(
        repeat_idx=0,
        fold_idx=0,
        prepared_fold=prepared,
        model=fitted_model,
        model_type=model_type,
    )

    performance = calculate_prediction_performance(
        y_resid_true_mat=y_true_target[None, :],
        y_pred_mat=y_pred_target[None, :],
    )
    observed_r, null_scores, p_result = fixed_prediction_permutation(
        y_true_target,
        y_pred_target,
        n_permutations=N_PERMUTATIONS,
        random_state=RANDOM_STATE + 101,
    )
    ci = bootstrap_metric_ci(
        y_true_target,
        y_pred_target,
        n_bootstrap=N_BOOTSTRAP,
        random_state=RANDOM_STATE + 202,
    )

    model_bundle = {
        "source_group": source_name,
        "target_group": target_name,
        "model_type": model_type,
        "model_params": best_model_params,
        "source_loocv_score": best_score,
        "source_loocv_candidate_scores": candidate_scores,
        "source_loocv_n_splits_actual": inner_n_splits_actual,
        "selection_params": SELECTION_PARAMS,
        "standardize_features": STANDARDIZE_FEATURES,
        "edge_record": edge_record,
        "fitted_model": fitted_model,
        "beta_y_source": beta_y_source,
        "covariate_names": covariate_names,
        "site_categories": site_categories,
    }

    prediction = {
        "ID": target["ID"],
        "y_true_raw": target["y"],
        "y_true_resid_source_transform": y_true_target,
        "y_pred_resid": y_pred_target,
    }

    print("Selected edges:", edge_record["edge_idx"].size)
    print("Performance:", performance["mean_prediction"])
    print("Permutation p:", p_result["p_value"])
    print("Bootstrap CI:", (ci["ci_low"], ci["ci_high"]))

    return model_bundle, prediction, performance, null_scores, p_result, ci


In [7]:
# ============================================================
# 6. 双向跨组预测
# ============================================================
low_to_high = fit_source_predict_target("lowIAT", "highIAT")
high_to_low = fit_source_predict_target("highIAT", "lowIAT")



SOURCE: lowIAT  ->  TARGET: highIAT
Locked by source LOOCV: {'ridge_alpha': 10.722672220103243}
Source LOOCV score: 0.28732097458278455
Selected edges: 74
Performance: {'pearson_r': 0.0267080648099148, 'r2': -5.2859102877545, 'mae': 20.284759449990922, 'rmse': 22.53158116576482}
Permutation p: 0.29154169166166766
Bootstrap CI: (-0.07199216365139847, 0.12924763017849356)

SOURCE: highIAT  ->  TARGET: lowIAT
Locked by source LOOCV: {'ridge_alpha': 3.5111917342151346}
Source LOOCV score: 0.2940970826069666
Selected edges: 323
Performance: {'pearson_r': -0.03753077146211042, 'r2': -8.226616572568796, 'mae': 23.26170038464101, 'rmse': 26.61056568683492}
Permutation p: 0.8322335532893421
Bootstrap CI: (-0.11827155687428127, 0.042720361544748946)


In [8]:
# ============================================================
# 7. 保存结果
# ============================================================
all_results = {
    "lowIAT_to_highIAT": low_to_high,
    "highIAT_to_lowIAT": high_to_low,
}

summary_rows = []
for direction, (model_bundle, prediction, performance, null_scores, p_result, ci) in all_results.items():
    source_name = model_bundle["source_group"]
    target_name = model_bundle["target_group"]
    prefix = f"{source_name}_to_{target_name}"

    with open(RESULT_DIR / f"{prefix}_final_model_bundle.pkl", "wb") as f:
        pickle.dump(model_bundle, f)
    with open(RESULT_DIR / f"{prefix}_prediction.pkl", "wb") as f:
        pickle.dump(prediction, f)
    with open(RESULT_DIR / f"{prefix}_performance.pkl", "wb") as f:
        pickle.dump(performance, f)

    np.savez_compressed(
        RESULT_DIR / f"{prefix}_inference_distributions.npz",
        permutation_null=null_scores,
        bootstrap_metric=ci["bootstrap_values"],
    )

    pred_df = pd.DataFrame({
        "ID": prediction["ID"],
        "source_group": source_name,
        "target_group": target_name,
        "y_true_raw": prediction["y_true_raw"],
        "y_true_resid_source_transform": prediction["y_true_resid_source_transform"],
        "y_pred_resid": prediction["y_pred_resid"],
    })
    pred_df.to_csv(RESULT_DIR / f"{prefix}_predictions.csv", index=False)

    perf = performance["mean_prediction"]
    summary_rows.append({
        "direction": direction,
        "source_group": source_name,
        "target_group": target_name,
        "n_source": int(groups[source_name]["y"].size),
        "n_target": int(groups[target_name]["y"].size),
        "n_edges_total": int(groups[source_name]["X"].shape[1]),
        "n_edges_selected": int(model_bundle["edge_record"]["edge_idx"].size),
        "ridge_alpha_locked_by_source_loocv": float(model_bundle["model_params"]["ridge_alpha"]),
        "source_loocv_pearson_r": float(model_bundle["source_loocv_score"]),
        "external_pearson_r": float(perf["pearson_r"]),
        "external_r2": float(perf["r2"]),
        "external_mae": float(perf["mae"]),
        "external_rmse": float(perf["rmse"]),
        "external_pearson_perm_p_greater": float(p_result["p_value"]),
        "external_pearson_ci_low": float(ci["ci_low"]),
        "external_pearson_ci_high": float(ci["ci_high"]),
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = RESULT_DIR / "CPM_nestedCV_crossGroup_summary.csv"
summary_df.to_csv(summary_path, index=False)

display(summary_df)
print("Saved summary:", summary_path)
print("Saved all results to:", RESULT_DIR)


,direction,source_group,target_group,n_source,n_target,n_edges_total,n_edges_selected,ridge_alpha_locked_by_source_loocv,source_loocv_pearson_r,external_pearson_r,external_r2,external_mae,external_rmse,external_pearson_perm_p_greater,external_pearson_ci_low,external_pearson_ci_high
0,lowIAT_to_highIAT,lowIAT,highIAT,653,461,22791,74,10.722672,0.287321,0.026708,-5.285910,20.284759,22.531581,0.291542,-0.071992,0.129248
1,highIAT_to_lowIAT,highIAT,lowIAT,461,653,22791,323,3.511192,0.294097,-0.037531,-8.226617,23.261700,26.610566,0.832234,-0.118272,0.042720


Saved summary: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_crossGroup/CPM_nestedCV_crossGroup_summary.csv
Saved all results to: /Users/fmri24/Desktop/YiZhang/CPM_nestedCV_crossGroup


## 8. 结果解释

主要看 `external_pearson_r` 和 `external_pearson_perm_p_greater`。

- `ridge_alpha_locked_by_source_loocv`：只在 source 组内部用 LOOCV 选出的最终 alpha；
- `source_loocv_pearson_r`：source 组内部锁定 alpha 时的 LOOCV 分数；
- `external_pearson_r`：source 最终模型预测 target 的残差尺度相关；
- `external_pearson_perm_p_greater`：固定 source 模型后，在 target 组随机打乱真实残差得到的单侧 permutation p 值。

如果跨组结果不显著，只能说明“当前 source 模型没有显示出稳定跨组泛化”；它不能直接证明两组机制完全不同。若要证明组内模型显著优于跨组模型，需要在同一交叉验证框架下直接比较两种预测方式。
